In [ ]:
import json
import mysql.connector
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# ONNX specific imports
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


def get_connection():
  return mysql.connector.connect(
      host="localhost",
      user="root",
      password="",
      database="marketdata",
  )


# ------------------------------------------
# 1. FETCH DATA FROM MYSQL WITH CONSTRAINT
# ------------------------------------------
try:
  db = get_connection()
  query = 'SELECT * FROM `labeled_flatdata` WHERE `edited` = "1"'
  df = pd.read_sql(query, db)
  print(
      f"Successfully loaded {len(df)} filtered rows (edited = '1') from MySQL"
      " database."
  )
except Exception as e:
  print(f"Database connection or query failed: {e}")
  df = pd.DataFrame()
finally:
  if "db" in locals() and db.is_connected():
    db.close()


# ------------------------------------------
# 2. PROCEED ONLY IF DATA IS LOADED
# ------------------------------------------
if df.empty:
  print(
      "DataFrame is empty. Check your database connection, credentials, and"
      " whether any rows have edited = '1'."
  )
else:
  annotation_columns = [
      "decision",
  ]

  valid_targets = [col for col in annotation_columns if col in df.columns]
  print(f"Valid target columns found: {valid_targets}\n")

  # ------------------------------------------
  # 3. INPUT FEATURES: M1 PRICE COLUMNS
  # ------------------------------------------
  feature_columns = [
      col
      for col in df.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  X = df[feature_columns].copy()
  X = X.apply(pd.to_numeric, errors="coerce")
  X = X.replace([np.inf, -np.inf], np.nan)

  # ------------------------------------------
  # 4. TRAIN, EXPORT ONNX, & SAVE CLASSES JSON
  # ------------------------------------------
  models = {}
  label_encoders = {}
  results = {}

  for target in valid_targets:
    y_series = df[target].fillna("").astype(str).str.strip()
    valid = (
        df[target].notna()
        & y_series.ne("")
        & y_series.ne("nan")
        & y_series.ne("None")
        & y_series.ne("NULL")
    )

    X_target = X.loc[valid]
    y_raw = y_series.loc[valid]

    print(f"Target '{target}': found {len(y_raw)} valid labeled rows.")

    if len(y_raw) < 2:
      print(
          f"Skipping {target}: insufficient labeled rows ({len(y_raw)} found,"
          " need at least 2)."
      )
      continue

    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)

    if len(encoder.classes_) < 2:
      print(
          f"Skipping {target}: only one unique label found ({encoder.classes_}).\n"
      )
      continue

    class_counts = pd.Series(y).value_counts()
    stratify = y if class_counts.min() >= 2 else None
    test_size = 0.2 if len(X_target) >= 5 else 0.5

    X_train, X_test, y_train, y_test = train_test_split(
        X_target,
        y,
        test_size=test_size,
        random_state=42,
        stratify=stratify,
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    models[target] = model
    label_encoders[target] = encoder

    results[target] = {
        "accuracy": accuracy,
        "training_rows": len(X_train),
        "testing_rows": len(X_test),
        "classes": list(encoder.classes_),
    }

    print(f"\n{'=' * 50}")
    print(f"TARGET: {target}")
    print(f"Accuracy: {accuracy:.2%}")

    # 1. Export ONNX Model
    try:
      initial_type = [("float_input", FloatTensorType([None, X_target.shape[1]]))]
      onnx_model = convert_sklearn(model, initial_types=initial_type)

      onnx_filename = f"model_{target}.onnx"
      with open(onnx_filename, "wb") as f:
        f.write(onnx_model.SerializeToString())

      # 2. Save corresponding class labels JSON
      classes_filename = f"classes_{target}.json"
      with open(classes_filename, "w") as f:
        json.dump(list(encoder.classes_), f)

      print(f"Successfully exported '{onnx_filename}' and '{classes_filename}'")
    except Exception as e:
      print(f"Failed to export {target}: {e}")

    print("-" * 50 + "\n")

  if results:
    results_df = pd.DataFrame(results).T
    display(results_df)

In [2]:
pip install mysql-connector-python numpy pandas scikit-learn skl2onnx onnx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import mysql.connector
import numpy as np
import onnxruntime as ort
import pandas as pd


# Helper to connect to DB for fetching new input data
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="marketdata",
        use_pure=True,
    )


# List of all annotation targets
target_models = [

    "decision",
]

# ---------------------------------------------------------
# 1. FETCH RECORD FROM DATABASE ONCE
# ---------------------------------------------------------
db = get_connection()
cur = db.cursor()
cur.execute("SELECT * FROM `labeled_flatdata` WHERE `id` = 10")
rows = cur.fetchall()
cols = [c[0] for c in cur.description]
cur.close()
db.close()

df_new = pd.DataFrame(rows, columns=cols)

if df_new.empty:
  print("No record found with id = 18.")
else:
  print(f"Successfully loaded record ID 18 for inference.\n")

  # ---------------------------------------------------------
  # 2. EXTRACT AND FORMAT FEATURES ONCE
  # ---------------------------------------------------------
  feature_columns = [
      col
      for col in df_new.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  X_new = df_new[feature_columns].copy()
  X_new = X_new.apply(pd.to_numeric, errors="coerce")
  X_new = X_new.replace([np.inf, -np.inf], np.nan)

  # Convert DataFrame row to a 2D numpy array of type float32
  input_data = X_new.to_numpy(dtype=np.float32)

  print("=" * 60)
  print("BATCH PREDICTION RESULTS FOR ROW ID 18")
  print("=" * 60)

  # ---------------------------------------------------------
  # 3. LOOP THROUGH EACH MODEL AND RUN INFERENCE
  # ---------------------------------------------------------
  for target_name in target_models:
    onnx_model_path = f"model_{target_name}.onnx"
    classes_path = f"classes_{target_name}.json"

    # Check if model and class files exist (skips if training skipped it)
    if not os.path.exists(onnx_model_path) or not os.path.exists(classes_path):
      print(
          f"[{target_name.upper()}] -> Skipped (Model or class JSON file not"
          " found)."
      )
      continue

    try:
      # Load ONNX session & label classes
      session = ort.InferenceSession(onnx_model_path)
      with open(classes_path, "r") as f:
        class_labels = json.load(f)

      input_name = session.get_inputs()[0].name
      expected_feature_count = session.get_inputs()[0].shape[1]

      # Safety check on feature dimensions
      if input_data.shape[1] != expected_feature_count:
        print(
            f"[{target_name.upper()}] -> Skipped (Feature mismatch: expected"
            f" {expected_feature_count}, got {input_data.shape[1]})."
        )
        continue

      # Run Inference
      outputs = session.run(None, {input_name: input_data})
      predicted_class_idx = outputs[0][0]
      class_probabilities = outputs[1]

      predicted_label = class_labels[int(predicted_class_idx)]

      print(f"Target: {target_name}")
      print(f"  -> Predicted Label : {predicted_label}")
      print(f"  -> Probabilities   : {class_probabilities}")
      print("-" * 60)

    except Exception as e:
      print(f"[{target_name.upper()}] -> Error during inference: {e}")
      print("-" * 60)

Successfully loaded record ID 18 for inference.

BATCH PREDICTION RESULTS FOR ROW ID 18
[DECISION] -> Skipped (Model or class JSON file not found).


In [4]:

import os
import json
import mysql.connector
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


# =========================================================
# CONFIGURATION
# =========================================================

TARGET = "decision"

OUTPUT_DIR = "training_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 1. DATABASE CONNECTION
# =========================================================

def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="marketdata",
        use_pure=True
    )


db = get_connection()

df = pd.read_sql(
    "SELECT * FROM labeled_flatdata ORDER BY id ASC",
    db
)

db.close()

print("Loaded records:", len(df))


# =========================================================
# 2. FEATURE EXTRACTION
# =========================================================

feature_columns = [
    col for col in df.columns
    if col.startswith("M1_")
    and not col.endswith("_time")
    and col.rsplit("_", 1)[-1]
    in {"open", "high", "low", "close"}
]

if TARGET not in df.columns:
    raise ValueError(f"Target column {TARGET} not found")

if not feature_columns:
    raise ValueError("No OHLC features found")

print("Features:", feature_columns)

# Convert features to numerical values
X = df[feature_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

X = X.replace([np.inf, -np.inf], np.nan)

# Remove invalid rows consistently
valid = X.notna().all(axis=1) & df[TARGET].notna()

X = X.loc[valid].reset_index(drop=True)

df = df.loc[valid].reset_index(drop=True)

y_original = df[TARGET].astype(str)


# =========================================================
# 3. ENCODE TARGET LABELS
# =========================================================

encoder = LabelEncoder()

y = encoder.fit_transform(y_original)

class_labels = encoder.classes_.tolist()

print("Classes:", class_labels)


# =========================================================
# 4. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# =========================================================

n = len(X)

if n < 10:
    raise ValueError("Not enough records for training")

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y[val_end:]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


# =========================================================
# 5. TRAIN RANDOM FOREST
# =========================================================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("\nTraining completed")


# =========================================================
# 6. VALIDATION AND TEST PREDICTIONS
# =========================================================

y_val_pred = model.predict(X_val)
y_pred = model.predict(X_test)

print("\nValidation accuracy:",
      accuracy_score(y_val, y_val_pred))

print("\nTest accuracy:",
      accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(class_labels)),
    target_names=class_labels,
    zero_division=0
))


# =========================================================
# 7. VISUALIZATION 1: CLASS DISTRIBUTION
# =========================================================

fig = px.histogram(
    df,
    x=TARGET,
    color=TARGET,
    title="Market Decision Distribution",
    text_auto=True
)

fig.write_html(
    f"{OUTPUT_DIR}/01_class_distribution.html"
)


# =========================================================
# 8. VISUALIZATION 2: MARKET DATA DISTRIBUTION
# =========================================================

fig = px.scatter(
    df,
    x="M1_open",
    y="M1_close",
    color=TARGET,
    hover_data=feature_columns,
    opacity=0.65,
    title="Market Data: Open vs Close"
)

fig.write_html(
    f"{OUTPUT_DIR}/02_market_distribution.html"
)


# =========================================================
# 9. VISUALIZATION 3: TRAIN / VALIDATION / TEST
# =========================================================

split_df = pd.DataFrame({
    "Record": np.arange(n),
    "Dataset": (
        ["Training"] * len(X_train)
        + ["Validation"] * len(X_val)
        + ["Testing"] * len(X_test)
    )
})

fig = px.scatter(
    split_df,
    x="Record",
    y="Dataset",
    color="Dataset",
    title="Chronological Training Data Split"
)

fig.write_html(
    f"{OUTPUT_DIR}/03_data_split.html"
)


# =========================================================
# 10. VISUALIZATION 4: CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(class_labels))
)

fig = px.imshow(
    cm,
    x=class_labels,
    y=class_labels,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Actual vs Predicted Confusion Matrix"
)

fig.update_layout(
    xaxis_title="Predicted",
    yaxis_title="Actual"
)

fig.write_html(
    f"{OUTPUT_DIR}/04_confusion_matrix.html"
)


# =========================================================
# 11. VISUALIZATION 5: ACTUAL VS PREDICTED
# =========================================================

comparison = pd.DataFrame({
    "Record": np.arange(len(y_test)),
    "Actual": encoder.inverse_transform(y_test),
    "Predicted": encoder.inverse_transform(y_pred)
})

comparison["Result"] = np.where(
    comparison["Actual"] == comparison["Predicted"],
    "Correct",
    "Incorrect"
)

fig = px.scatter(
    comparison,
    x="Record",
    y="Actual",
    color="Result",
    hover_data=["Predicted"],
    title="Actual vs Predicted Test Observations"
)

fig.write_html(
    f"{OUTPUT_DIR}/05_predictions.html"
)


# =========================================================
# 12. VISUALIZATION 6: FEATURE IMPORTANCE
# =========================================================

importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
}).sort_values("Importance")

fig = px.bar(
    importance,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Random Forest Feature Importance"
)

fig.write_html(
    f"{OUTPUT_DIR}/06_feature_importance.html"
)


# =========================================================
# 13. VISUALIZATION 7: PREDICTION CONFIDENCE
# =========================================================

probabilities = model.predict_proba(X_test)

confidence = pd.DataFrame({
    "Confidence": probabilities.max(axis=1),
    "Prediction": encoder.inverse_transform(y_pred)
})

fig = px.histogram(
    confidence,
    x="Confidence",
    color="Prediction",
    nbins=30,
    title="Prediction Confidence Distribution"
)

fig.write_html(
    f"{OUTPUT_DIR}/07_prediction_confidence.html"
)


# =========================================================
# 14. VISUALIZATION 8: PREDICTION PROBABILITIES
# =========================================================

probability_df = pd.DataFrame(
    probabilities,
    columns=class_labels
)

probability_df["Record"] = np.arange(len(X_test))

fig = go.Figure()

for label in class_labels:
    fig.add_trace(
        go.Scatter(
            x=probability_df["Record"],
            y=probability_df[label],
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Prediction Probabilities Across Test Data",
    xaxis_title="Test Record",
    yaxis_title="Probability",
    yaxis=dict(range=[0, 1])
)

fig.write_html(
    f"{OUTPUT_DIR}/08_probability_timeline.html"
)


# =========================================================
# 15. EXPORT ONNX MODEL
# =========================================================

initial_type = [
    (
        "float_input",
        FloatTensorType([None, len(feature_columns)])
    )
]

onnx_model = convert_sklearn(
    model,
    initial_types=initial_type
)

with open("model_decision.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())


# =========================================================
# 16. EXPORT CLASS LABELS
# =========================================================

with open("classes_decision.json", "w") as f:
    json.dump(class_labels, f)


# =========================================================
# 17. SAVE FEATURE ORDER
# =========================================================

with open("features_decision.json", "w") as f:
    json.dump(feature_columns, f)


# =========================================================
# 18. FINAL RESULTS
# =========================================================

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

print("Test accuracy:", accuracy_score(y_test, y_pred))

print("Classes:", class_labels)

print("ONNX model: model_decision.onnx")
print("Labels: classes_decision.json")
print("Feature order: features_decision.json")

print("Visualizations:", OUTPUT_DIR)

print("\nOpen the HTML files in Chrome to explore your charts.")


C:\Users\nishw\AppData\Local\Temp\ipykernel_20180\410042052.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Loaded records: 283
Features: ['M1_01_open', 'M1_01_high', 'M1_01_low', 'M1_01_close', 'M1_02_open', 'M1_02_high', 'M1_02_low', 'M1_02_close', 'M1_03_open', 'M1_03_high', 'M1_03_low', 'M1_03_close', 'M1_04_open', 'M1_04_high', 'M1_04_low', 'M1_04_close', 'M1_05_open', 'M1_05_high', 'M1_05_low', 'M1_05_close', 'M1_06_open', 'M1_06_high', 'M1_06_low', 'M1_06_close', 'M1_07_open', 'M1_07_high', 'M1_07_low', 'M1_07_close', 'M1_08_open', 'M1_08_high', 'M1_08_low', 'M1_08_close', 'M1_09_open', 'M1_09_high', 'M1_09_low', 'M1_09_close', 'M1_10_open', 'M1_10_high', 'M1_10_low', 'M1_10_close', 'M1_11_open', 'M1_11_high', 'M1_11_low', 'M1_11_close', 'M1_12_open', 'M1_12_high', 'M1_12_low', 'M1_12_close', 'M1_13_open', 'M1_13_high', 'M1_13_low', 'M1_13_close', 'M1_14_open', 'M1_14_high', 'M1_14_low', 'M1_14_close', 'M1_15_open', 'M1_15_high', 'M1_15_low', 'M1_15_close', 'M1_16_open', 'M1_16_high', 'M1_16_low', 'M1_16_close', 'M1_17_open', 'M1_17_high', 'M1_17_low', 'M1_17_close', 'M1_18_open', 'M1

ValueError: Not enough records for training

In [6]:

import os
import json
import mysql.connector
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


# =========================================================
# CONFIGURATION
# =========================================================

TARGET = "decision"

OUTPUT_DIR = "training_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# 1. DATABASE CONNECTION
# =========================================================

def get_connection():

    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="marketdata",
        use_pure=True
    )


db = get_connection()

df = pd.read_sql(
    """
    SELECT *
    FROM labeled_flatdata
    WHERE edited = 1
    ORDER BY id ASC
    """,
    db
)

db.close()

print("=" * 60)
print("DATABASE LOADED")
print("=" * 60)

print("Total edited records:", len(df))


# =========================================================
# 2. EXTRACT OHLC FEATURES
# =========================================================

feature_columns = [
    col for col in df.columns
    if col.startswith("M1_")
    and not col.endswith("_time")
    and col.rsplit("_", 1)[-1]
    in {"open", "high", "low", "close"}
]

if not feature_columns:
    raise ValueError("No OHLC features found")

if TARGET not in df.columns:
    raise ValueError("Decision column not found")

print("\nTotal features:", len(feature_columns))


# =========================================================
# 3. CLEAN DECISION LABELS
# =========================================================

df[TARGET] = (
    df[TARGET]
    .fillna("none")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("", "none")
)

# Only the three intended classes
allowed_classes = ["buy", "sell", "none"]

df = df[
    df[TARGET].isin(allowed_classes)
].copy()

# Convert features to numeric
X = df[feature_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

X = X.replace([np.inf, -np.inf], np.nan)

# Remove records with invalid features
valid = X.notna().all(axis=1)

X = X.loc[valid].reset_index(drop=True)

df = df.loc[valid].reset_index(drop=True)

y_original = df[TARGET].reset_index(drop=True)

print("\nFINAL CLASS DISTRIBUTION")

print(y_original.value_counts())

print("\nTotal usable records:", len(X))


# =========================================================
# 4. VISUALIZATION: CLASS DISTRIBUTION
# =========================================================

distribution = (
    y_original.value_counts()
    .reindex(allowed_classes, fill_value=0)
    .rename_axis("Decision")
    .reset_index(name="Records")
)

fig = px.bar(
    distribution,
    x="Decision",
    y="Records",
    color="Decision",
    text="Records",
    title="BUY / SELL / NONE Distribution"
)

fig.write_html(
    f"{OUTPUT_DIR}/01_class_distribution.html"
)


# =========================================================
# 5. ENCODE TARGET LABELS
# =========================================================

encoder = LabelEncoder()

y = encoder.fit_transform(y_original)

class_labels = encoder.classes_.tolist()

print("\nEncoded classes:", class_labels)


# =========================================================
# 6. CHRONOLOGICAL DATA SPLIT
# =========================================================

n = len(X)

if n < 10:
    raise ValueError(
        f"Only {n} usable records. "
        "Collect more labeled data before training."
    )

train_end = int(n * 0.70)

val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y[val_end:]

print("\nDATA SPLIT")

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


# =========================================================
# 7. VISUALIZATION: TRAINING TIMELINE
# =========================================================

split_df = pd.DataFrame({
    "Record": np.arange(n),
    "Dataset": (
        ["Training"] * len(X_train)
        + ["Validation"] * len(X_val)
        + ["Testing"] * len(X_test)
    )
})

fig = px.scatter(
    split_df,
    x="Record",
    y="Dataset",
    color="Dataset",
    title="Chronological Training / Validation / Testing"
)

fig.write_html(
    f"{OUTPUT_DIR}/02_data_split.html"
)


# =========================================================
# 8. TRAIN MULTICLASS RANDOM FOREST
# =========================================================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    class_weight=None,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("\nMODEL TRAINING COMPLETED")


# =========================================================
# 9. VALIDATION
# =========================================================

y_val_pred = model.predict(X_val)

print("\nVALIDATION ACCURACY:")

print(
    accuracy_score(
        y_val,
        y_val_pred
    )
)


# =========================================================
# 10. TEST PREDICTIONS
# =========================================================

y_pred = model.predict(X_test)

print("\nTEST ACCURACY:")

print(
    accuracy_score(
        y_test,
        y_pred
    )
)


# =========================================================
# 11. CLASSIFICATION REPORT
# =========================================================

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(class_labels)),
        target_names=class_labels,
        zero_division=0
    )
)


# =========================================================
# 12. VISUALIZATION: CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(class_labels))
)

fig = px.imshow(
    cm,
    x=class_labels,
    y=class_labels,
    text_auto=True,
    color_continuous_scale="Blues",
    title="BUY / SELL / NONE Confusion Matrix"
)

fig.update_layout(
    xaxis_title="Predicted",
    yaxis_title="Actual"
)

fig.write_html(
    f"{OUTPUT_DIR}/03_confusion_matrix.html"
)


# =========================================================
# 13. VISUALIZATION: ACTUAL VS PREDICTED
# =========================================================

comparison = pd.DataFrame({
    "Record": np.arange(len(y_test)),
    "Actual": encoder.inverse_transform(y_test),
    "Predicted": encoder.inverse_transform(y_pred)
})

comparison["Result"] = np.where(
    comparison["Actual"] == comparison["Predicted"],
    "Correct",
    "Incorrect"
)

fig = px.scatter(
    comparison,
    x="Record",
    y="Actual",
    color="Result",
    symbol="Result",
    hover_data=["Predicted"],
    title="Actual vs Predicted Decisions"
)

fig.write_html(
    f"{OUTPUT_DIR}/04_actual_vs_predicted.html"
)


# =========================================================
# 14. VISUALIZATION: FEATURE IMPORTANCE
# =========================================================

importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
}).sort_values(
    "Importance",
    ascending=True
)

fig = px.bar(
    importance,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Random Forest Feature Importance"
)

fig.write_html(
    f"{OUTPUT_DIR}/05_feature_importance.html"
)


# =========================================================
# 15. VISUALIZATION: PREDICTION CONFIDENCE
# =========================================================

probabilities = model.predict_proba(X_test)

confidence = pd.DataFrame({
    "Confidence": probabilities.max(axis=1),
    "Prediction": encoder.inverse_transform(y_pred)
})

fig = px.histogram(
    confidence,
    x="Confidence",
    color="Prediction",
    nbins=30,
    title="Prediction Confidence Distribution"
)

fig.write_html(
    f"{OUTPUT_DIR}/06_prediction_confidence.html"
)


# =========================================================
# 16. VISUALIZATION: CLASS PROBABILITIES OVER TIME
# =========================================================

probability_df = pd.DataFrame(
    probabilities,
    columns=class_labels
)

probability_df["Record"] = np.arange(
    len(probability_df)
)

fig = go.Figure()

for label in class_labels:

    fig.add_trace(
        go.Scatter(
            x=probability_df["Record"],
            y=probability_df[label],
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="BUY / SELL / NONE Probabilities",
    xaxis_title="Test Observation",
    yaxis_title="Probability",
    yaxis=dict(range=[0, 1])
)

fig.write_html(
    f"{OUTPUT_DIR}/07_probability_timeline.html"
)


# =========================================================
# 17. VISUALIZATION: MARKET OHLC DISTRIBUTION
# =========================================================

if "M1_01_open" in df.columns and "M1_01_close" in df.columns:

    fig = px.scatter(
        df,
        x="M1_01_open",
        y="M1_01_close",
        color=TARGET,
        hover_data=["id"],
        title="Market Open vs Close by Decision"
    )

    fig.write_html(
        f"{OUTPUT_DIR}/08_market_distribution.html"
    )


# =========================================================
# 18. EXPORT ONNX MODEL
# =========================================================

initial_type = [
    (
        "float_input",
        FloatTensorType([
            None,
            len(feature_columns)
        ])
    )
]

onnx_model = convert_sklearn(
    model,
    initial_types=initial_type
)

with open(
    "model_decision.onnx",
    "wb"
) as f:

    f.write(
        onnx_model.SerializeToString()
    )


# =========================================================
# 19. EXPORT CLASS LABELS
# =========================================================

with open(
    "classes_decision.json",
    "w"
) as f:

    json.dump(
        class_labels,
        f
    )


# =========================================================
# 20. EXPORT FEATURE ORDER
# =========================================================

with open(
    "features_decision.json",
    "w"
) as f:

    json.dump(
        feature_columns,
        f
    )


# =========================================================
# 21. FINAL SUMMARY
# =========================================================

print("\n" + "=" * 60)

print("TRAINING COMPLETED")

print("=" * 60)

print("Training records:", len(X_train))

print("Validation records:", len(X_val))

print("Testing records:", len(X_test))

print("Features:", len(feature_columns))

print("Classes:", class_labels)

print(
    "Test accuracy:",
    accuracy_score(y_test, y_pred)
)

print("\nEXPORTED FILES:")

print("model_decision.onnx")

print("classes_decision.json")

print("features_decision.json")

print("\nVISUALIZATION FOLDER:")

print(OUTPUT_DIR)

print("\nOpen the HTML files in Chrome.")


C:\Users\nishw\AppData\Local\Temp\ipykernel_20180\3223993819.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


DATABASE LOADED
Total edited records: 174

Total features: 80

FINAL CLASS DISTRIBUTION
decision
none    168
buy       5
sell      1
Name: count, dtype: int64

Total usable records: 174

Encoded classes: ['buy', 'none', 'sell']

DATA SPLIT
Training: 121
Validation: 26
Testing: 27

MODEL TRAINING COMPLETED

VALIDATION ACCURACY:
1.0

TEST ACCURACY:
0.8888888888888888

CLASSIFICATION REPORT
              precision    recall  f1-score   support

         buy       0.00      0.00      0.00         3
        none       0.89      1.00      0.94        24
        sell       0.00      0.00      0.00         0

    accuracy                           0.89        27
   macro avg       0.30      0.33      0.31        27
weighted avg       0.79      0.89      0.84        27


TRAINING COMPLETED
Training records: 121
Validation records: 26
Testing records: 27
Features: 80
Classes: ['buy', 'none', 'sell']
Test accuracy: 0.8888888888888888

EXPORTED FILES:
model_decision.onnx
classes_decision.json
feat